In [1]:
import torch
print("CUDA beschikbaar:", torch.cuda.is_available())
print("GPU naam:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "geen GPU")

CUDA beschikbaar: True
GPU naam: Tesla T4


In [3]:
!pip install -q transformers

In [4]:
import pandas as pd

URL = "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/BBBP.csv"
df = pd.read_csv(URL)

print("Aantal rijen:", len(df))
print("Kolommen:", df.columns.tolist())
df.head()

Aantal rijen: 2050
Kolommen: ['num', 'name', 'p_np', 'smiles']


,num,name,p_np,smiles
0,1,Propanolol,1,[Cl].CC(C)NCC(O)COc1cccc2ccccc12
1,2,Terbutylchlorambucil,1,C(=O)(OC(C)(C)C)CCCc1ccc(cc1)N(CCCl)CCCl
2,3,40730,1,c12c3c(N4CCN(C)CC4)c(F)cc1c(c(C(O)=O)cn2C(C)CO...
3,4,24,1,C1CCN(CC1)Cc1cccc(c1)OCCCNC(=O)C
4,5,cloxacillin,1,Cc1onc(c2ccccc2Cl)c1C(=O)N[C@H]3[C@H]4SC(C)(C)...


In [5]:
print(df["p_np"].value_counts())
print("\nFractie positief:", df["p_np"].mean().round(3))

p_np
1    1567
0     483
Name: count, dtype: int64

Fractie positief: 0.764


In [6]:
print("Missende SMILES:", df["smiles"].isna().sum())
df = df.dropna(subset=["smiles"]).reset_index(drop=True)

Missende SMILES: 0


In [7]:
from sklearn.model_selection import train_test_split

smiles = df["smiles"].tolist()
labels = df["p_np"].astype(int).tolist()

# Eerst test eraf halen (10%)
train_smiles, test_smiles, train_labels, test_labels = train_test_split(
    smiles, labels, test_size=0.1, random_state=42, stratify=labels
)
# Daarna val van train (10% van de rest)
train_smiles, val_smiles, train_labels, val_labels = train_test_split(
    train_smiles, train_labels, test_size=0.1, random_state=42, stratify=train_labels
)

print(f"Train: {len(train_smiles)} | Val: {len(val_smiles)} | Test: {len(test_smiles)}")

Train: 1660 | Val: 185 | Test: 205


In [8]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "DeepChem/ChemBERTa-77M-MLM"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/631 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/420 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/13.7M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/53 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: DeepChem/ChemBERTa-77M-MLM
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [9]:
import torch

# Bereken gewichten op basis van je trainset
n_neg = train_labels.count(0)   # 483-ish
n_pos = train_labels.count(1)   # 1567-ish
weight_neg = n_pos / n_neg      # ≈ 3.24
weight_pos = 1.0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_weights = torch.tensor([weight_neg, weight_pos]).to(device)
print(f"Class weights: BBB- = {weight_neg:.2f}, BBB+ = {weight_pos:.2f}")

Class weights: BBB- = 3.25, BBB+ = 1.00


In [10]:
sample = "CC(=O)Oc1ccccc1C(=O)O"  # aspirine
tokens = tokenizer.tokenize(sample)
print("SMILES:", sample)
print("Tokens:", tokens)
print("Aantal tokens:", len(tokens))


SMILES: CC(=O)Oc1ccccc1C(=O)O
Tokens: ['C', 'C', '(', '=', 'O', ')', 'O', 'c', '1', 'c', 'c', 'c', 'c', 'c', '1', 'C', '(', '=', 'O', ')', 'O']
Aantal tokens: 21


In [11]:
import torch
from torch.utils.data import Dataset

class SMILESDataset(Dataset):
    def __init__(self, smiles, labels, tokenizer, max_length=128):
        self.smiles = smiles
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.smiles[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].flatten(),
            "attention_mask": enc["attention_mask"].flatten(),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }

train_ds = SMILESDataset(train_smiles, train_labels, tokenizer)
val_ds   = SMILESDataset(val_smiles,   val_labels,   tokenizer)
test_ds  = SMILESDataset(test_smiles,  test_labels,  tokenizer)

print("Aantal samples in train_ds:", len(train_ds))
print("Voorbeeld input_ids shape:", train_ds[0]["input_ids"].shape)

Aantal samples in train_ds: 1660
Voorbeeld input_ids shape: torch.Size([128])


In [12]:
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    return {
        "accuracy": accuracy_score(labels, preds),
        "roc_auc":  roc_auc_score(labels, probs),
    }

In [13]:
from transformers import TrainingArguments, Trainer
from torch.nn import CrossEntropyLoss
from transformers import Trainer

training_args = TrainingArguments(
    output_dir="./chemberta_bbbp",
    num_train_epochs=10,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="roc_auc",
    greater_is_better=True,
    logging_steps=20,
    save_total_limit=2,
    report_to="none",
    fp16=True,            # mixed-precision: sneller op T4 GPU
)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = CrossEntropyLoss(weight=class_weights)(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

In [14]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Roc Auc
1,0.691015,0.671100,0.794595,0.781109
2,0.622471,0.592101,0.832432,0.911670
3,0.564465,0.499535,0.827027,0.936654
4,0.469158,0.414647,0.832432,0.942940
5,0.409512,0.357506,0.837838,0.947776
6,0.371232,0.323468,0.832432,0.952289
7,0.344699,0.306464,0.837838,0.953417
8,0.314308,0.296944,0.837838,0.954384
9,0.329021,0.291834,0.837838,0.956157
10,0.302910,0.290177,0.843243,0.955835


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.beta', 'roberta.embeddings.LayerNorm.gamma', 'roberta.encoder.layer.0.attention.output.LayerNorm.beta', 'roberta.enc

TrainOutput(global_step=520, training_loss=0.44379408542926496, metrics={'train_runtime': 24.0784, 'train_samples_per_second': 689.415, 'train_steps_per_second': 21.596, 'total_flos': 38242141900800.0, 'train_loss': 0.44379408542926496, 'epoch': 10.0})

In [15]:
test_metrics = trainer.evaluate(test_ds)

print("=== Testresultaten ===")
for k, v in test_metrics.items():
    if isinstance(v, float):
        print(f"{k:25s}: {v:.4f}")

=== Testresultaten ===
eval_loss                : 0.2548
eval_accuracy            : 0.8683
eval_roc_auc             : 0.9700
eval_runtime             : 0.1106
eval_samples_per_second  : 1853.7060
eval_steps_per_second    : 36.1700
epoch                    : 10.0000


In [16]:
from sklearn.metrics import confusion_matrix, classification_report

# Voorspellingen op de test-set
predictions = trainer.predict(test_ds)
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
print("Confusion matrix:")
print("                 voorspeld 0   voorspeld 1")
print(f"  werkelijk 0:    {cm[0,0]:5d}        {cm[0,1]:5d}")
print(f"  werkelijk 1:    {cm[1,0]:5d}        {cm[1,1]:5d}")

print("\nPer-klasse metrics:")
print(classification_report(y_true, y_pred, target_names=["geen BBB (0)", "wel BBB (1)"]))

Confusion matrix:
                 voorspeld 0   voorspeld 1
  werkelijk 0:       47            1
  werkelijk 1:       26          131

Per-klasse metrics:
              precision    recall  f1-score   support

geen BBB (0)       0.64      0.98      0.78        48
 wel BBB (1)       0.99      0.83      0.91       157

    accuracy                           0.87       205
   macro avg       0.82      0.91      0.84       205
weighted avg       0.91      0.87      0.88       205

